# Parallel Cleaning

After viewing the raw text files, we notice that the three text files follow similar formatting and we try to clean them in parallel by applying the same steps.

1. Remove metadata and text that are not verses.
2. Compile verses separated into new lines into one. (Now we have verse segmentation yay)
3. From verses, segment verses into their own sentences. 

In [678]:
import regex as re

with open('cebuano_raw.txt', 'r') as file:
    cebuano_raw = file.read()

with open('filipino_raw.txt', 'r') as file:
    filipino_raw = file.read()

with open('hiligaynon_raw.txt', 'r') as file:
    hiligaynon_raw = file.read()

In [679]:
texts = {'ceb' : cebuano_raw, 'fil': filipino_raw, 'hil' : hiligaynon_raw}

def sample(n):
    for key in texts.keys():
        print(key.capitalize())
        print(texts[key][0:n])

def save():
    for key in texts.keys():
        with open(f"checkpoints/{key}_clean.txt", 'w') as f:
            f.write(texts[key])
            

#we use this function as a sanity check to validate whether all our texts contain the same number of lines. As our goal is each verse/sentence separated by new line, having the same number of lines across all 3 strings is a good indicator we have parallel data.
def print_number_of_lines():
    print('number of lines:')
    for key in texts.keys():
        print(key, ':', texts[key].count('\n'))

print_number_of_lines()

number of lines:
ceb : 97080
fil : 95169
hil : 102212


## Removing Metadata

metadata includes page headers, page numbers, table of contents, chapter titles

### Title Page, Table of Contents, Esclarecimento

We noticed that the three pages contain all of these unnecessary sections, in this part we try to remove them from the text files

In [680]:
#Esclarecimento is a clarification section in portuguese, since it is in the last section, we can safely remove all the text after this word.

disclaimer = re.compile(r'((?:# Esclarecimento|\*\*Esclarecimento\*\*).*)', flags=re.DOTALL) #we discard all lines after esclarecimento

for key in texts.keys():
    texts[key] = re.sub(disclaimer, '', texts[key])

print_number_of_lines()

number of lines:
ceb : 97042
fil : 95142
hil : 102173


cross checking the number of lines with the actual text files, we can say that we actually removed the last section. We can apply the same logic to the title page and table of contents, discarding all texts before `Genesis 1`, as this indicates the start of the first chapter of the first book, all text above it can be discarded. 

In [681]:
genesis = re.compile(r'^(.*?)(?:# Genesis\n|Genesis\n)', flags=re.DOTALL)

for key in texts.keys():
    texts[key] = re.sub(genesis, '', texts[key])
    print(texts[key][0:4000])

print_number_of_lines()




**Genesis 1**
Ang Sugilanon sa Kabuhatan

(^1) Sa pagsugod sa Dios pagbuhat sa kalibotan ug sa tanang butang sa
kalangitan,
(^2) ang kalibotan haw-ang ug walay hitsura. Ang kadagatan gitabonan sa
kangitngit ug ang Espiritu sa Dios naglihok ibabaw sa katubigan.
(^3) Unya miingon ang Dios, “Motungha ang kahayag.” Ug mitungha ang
kahayag.
(^4) Nahimuot ang Dios sa kahayag nga iyang nakita ug gilain niya kini gikan sa
kangitngit.
(^5) Ginganlan niyag “Adlaw” ang kahayag ug ang kangitngit ginganlan niyag
“Gabii.” Milabay ang kagabhion ug miabot ang kabuntagon. Mao kadto ang
unang adlaw.
(^6) Unya miingon ang Dios, “Motungha ang usa ka wanang nga magbahin sa
katubigan.” Ug nahimo kini.
(^7) Pinaagi sa gibuhat niyang wanang, gilain sa Dios ang tubig nga didto sa
ibabaw gikan sa tubig nga dinhi sa ubos ug nahimo kini.
(^8) Ginganlan niyag “Langit” ang wanang. Milabay na usab ang kagabhion ug
miabot ang kabuntagon. Mao kadto ang ikaduhang adlaw.
(^9) Unya miingon ang Dios, “Matingob sa usa lam

as we can see, all of the texts now start at Genesis 1, for the last part, let's remove the page footers:

In [682]:
fil_footer = r'\n```\nMagandang Balita Biblia Revised\n```\n'
hil_footer = r'\n```\nMaayong Balita nga Biblia\n© 2012 Philippine Bible Society.\n```\n'
ceb_footer = r'\n```\nAng Bag-ong Maayong Balita Biblia\n```\n'

texts['fil'] = re.sub(fil_footer, '', texts['fil'], flags=re.DOTALL)
texts['hil'] = re.sub(hil_footer, '', texts['hil'], flags=re.DOTALL)
texts['ceb'] = re.sub(ceb_footer, '', texts['ceb'], flags=re.DOTALL)

print_number_of_lines()

sample(10000)

number of lines:
ceb : 85478
fil : 83874
hil : 89943
Ceb

**Genesis 1**
Ang Sugilanon sa Kabuhatan

(^1) Sa pagsugod sa Dios pagbuhat sa kalibotan ug sa tanang butang sa
kalangitan,
(^2) ang kalibotan haw-ang ug walay hitsura. Ang kadagatan gitabonan sa
kangitngit ug ang Espiritu sa Dios naglihok ibabaw sa katubigan.
(^3) Unya miingon ang Dios, “Motungha ang kahayag.” Ug mitungha ang
kahayag.
(^4) Nahimuot ang Dios sa kahayag nga iyang nakita ug gilain niya kini gikan sa
kangitngit.
(^5) Ginganlan niyag “Adlaw” ang kahayag ug ang kangitngit ginganlan niyag
“Gabii.” Milabay ang kagabhion ug miabot ang kabuntagon. Mao kadto ang
unang adlaw.
(^6) Unya miingon ang Dios, “Motungha ang usa ka wanang nga magbahin sa
katubigan.” Ug nahimo kini.
(^7) Pinaagi sa gibuhat niyang wanang, gilain sa Dios ang tubig nga didto sa
ibabaw gikan sa tubig nga dinhi sa ubos ug nahimo kini.
(^8) Ginganlan niyag “Langit” ang wanang. Milabay na usab ang kagabhion ug
miabot ang kabuntagon. Mao kadto ang ikaduhan

In [683]:
save()

after inspecting, the current state of the whole text files, we notice that the texts still have some metadata to mark the start of the new testament

In [684]:
ceb_new = r'\n# Cebuano - All Bible\n'
hil_new = r'\n# Hiligaynon - All Bible\n'
fil_new = r'\*\*Filipino - All Bible\*\*\n\(Magandang Balita Biblia Revised\)\nBAGONG TIPAN\n'

texts['fil'] = re.sub(fil_new, '', texts['fil'], flags=re.DOTALL)
texts['hil'] = re.sub(hil_new, '', texts['hil'], flags=re.DOTALL)
texts['ceb'] = re.sub(ceb_new, '', texts['ceb'], flags=re.DOTALL)

print_number_of_lines()
save()

number of lines:
ceb : 85476
fil : 83871
hil : 89941


In [685]:
#remove chapter names
chapters = re.compile(r'\*\*.*?\*\*\n', re.DOTALL)
for key in texts.keys():
    texts[key] = re.sub(chapters, '', texts[key])

sample(5000)
print_number_of_lines()

Ceb

Ang Sugilanon sa Kabuhatan

(^1) Sa pagsugod sa Dios pagbuhat sa kalibotan ug sa tanang butang sa
kalangitan,
(^2) ang kalibotan haw-ang ug walay hitsura. Ang kadagatan gitabonan sa
kangitngit ug ang Espiritu sa Dios naglihok ibabaw sa katubigan.
(^3) Unya miingon ang Dios, “Motungha ang kahayag.” Ug mitungha ang
kahayag.
(^4) Nahimuot ang Dios sa kahayag nga iyang nakita ug gilain niya kini gikan sa
kangitngit.
(^5) Ginganlan niyag “Adlaw” ang kahayag ug ang kangitngit ginganlan niyag
“Gabii.” Milabay ang kagabhion ug miabot ang kabuntagon. Mao kadto ang
unang adlaw.
(^6) Unya miingon ang Dios, “Motungha ang usa ka wanang nga magbahin sa
katubigan.” Ug nahimo kini.
(^7) Pinaagi sa gibuhat niyang wanang, gilain sa Dios ang tubig nga didto sa
ibabaw gikan sa tubig nga dinhi sa ubos ug nahimo kini.
(^8) Ginganlan niyag “Langit” ang wanang. Milabay na usab ang kagabhion ug
miabot ang kabuntagon. Mao kadto ang ikaduhang adlaw.
(^9) Unya miingon ang Dios, “Matingob sa usa lamang ka dap

In [686]:
#remove book names 
ceb_hil_books = r'(^# .*?$)'
fil_books = r"Genesis|Exodo|Levitico|Mga Bilang|Deuteronomio|Josue|Mga Hukom|Ruth|1 Samuel|2 Samuel|1 Mga Hari|2 Mga Hari|1 Mga Cronica|2 Mga Cronica|Ezra|Nehemias|Ester|Job|Mga Awit|Mga Kawikaan|Ang Mangangaral|Ang Awit ni Solomon|Isaias|Jeremias|Mga Panaghoy|Ezekiel|Daniel|Hosea|Joel|Amos|Obadias|Jonas|Mikas|Nahum|Habakuk|Zefanias|Hagai|Zacarias|Malakias|Mateo|Marcos|Lucas|Juan|Mga Gawa|Mga Taga-Roma|1 Mga Taga-Corinto|2 Mga Taga-Corinto|Mga Taga-Galacia|Mga Taga-Efeso|Mga Taga-Filipos|Mga Taga-Colosas|1 Mga Taga-Tesalonica|2 Mga Taga-Tesalonica|1 Timoteo|2 Timoteo|Tito|Filemon|Mga Hebreo|Santiago|1 Pedro|2 Pedro|1 Juan|2 Juan|3 Juan|Judas|Pahayag"

texts['fil'] = re.sub(fil_books, '', texts['fil'], flags=re.MULTILINE)
texts['hil'] = re.sub(ceb_hil_books, '', texts['hil'], flags=re.MULTILINE)
texts['ceb'] = re.sub(ceb_hil_books, '', texts['ceb'], flags=re.MULTILINE)

save()
print_number_of_lines()

number of lines:
ceb : 84214
fil : 82338
hil : 88263


In [687]:
#remove cross references (references to other bible verses)
cross_ref = r'^\(\w.*$'

texts['fil'] = re.sub(cross_ref, '', texts['fil'], flags=re.MULTILINE)
texts['hil'] = re.sub(cross_ref, '', texts['hil'], flags=re.MULTILINE)
texts['ceb'] = re.sub(cross_ref, '', texts['ceb'], flags=re.MULTILINE)

print_number_of_lines()
save()

number of lines:
ceb : 84214
fil : 82338
hil : 88263


(^) is persistent across all files so we can remove them. Then let's remove all empty lines

In [688]:
noisy_parse = r'\(\^\)'
empty_lines = r'^\s*\n'

texts['fil'] = re.sub(noisy_parse, '', texts['fil'], flags=re.MULTILINE)
texts['hil'] = re.sub(noisy_parse, '', texts['hil'], flags=re.MULTILINE)
texts['ceb'] = re.sub(noisy_parse, '', texts['ceb'], flags=re.MULTILINE)

texts['fil'] = re.sub(empty_lines, '', texts['fil'], flags=re.MULTILINE)
texts['hil'] = re.sub(empty_lines, '', texts['hil'], flags=re.MULTILINE)
texts['ceb'] = re.sub(empty_lines, '', texts['ceb'], flags=re.MULTILINE)

print_number_of_lines()
save()

number of lines:
ceb : 77497
fil : 75987
hil : 84509


Now it gets exciting, looking at our data right now, we see that verses begin with a number, let's merge lines that don't start with a number to the previous line


In [689]:
not_own_verse = r'\n(?!([\(\d]))'

texts['ceb'] = re.sub(not_own_verse, ' ', texts['ceb'])
texts['fil'] = re.sub(not_own_verse, ' ', texts['fil'])
texts['hil'] = re.sub(not_own_verse, ' ', texts['hil'])

print_number_of_lines()
save()

number of lines:
ceb : 30676
fil : 30421
hil : 30159


found an outlier in cebuano for 2 , 400

In [690]:
texts['ceb'] = re.sub(r'(\n)\(?\^?(\d{3}|\d{1,3} , \d{3}|\d+. \d+)\)?', r' \2', texts['ceb'])
texts['ceb'] = re.sub(r'\n\(\^5\) , \(\^000\)', r' 5 , 000', texts['ceb'])
texts['ceb'] = re.sub(r'(\d+.) (\d+)', r'\1\2', texts['ceb'])

print_number_of_lines()
save()

number of lines:
ceb : 30551
fil : 30421
hil : 30159


In [691]:
texts['ceb'] = re.sub(r'\n\(\^(70)\) \.(Niadtong higayona didto na si Jose sa Ehipto\.)', r' \1. \2', texts['ceb'])

now let's fix other numbers, such as this found in the text

```
(^144) , (^000) gikan sa tanang banay sa Israel.
(^5) Sa banay ni Juda
(^12) , (^000) ang napatikan; sa banay ni Ruben
(^12) , (^000) ; sa banay ni Gad
(^12) , (^000) ;
(^6) sa banay ni Aser
(^12) , (^000) ; sa banay ni Neftali
(^12) , (^000) ; sa banay ni Manases
(^12) , (^000) ;
(^7) sa banay ni Simeon
(^12) , (^000) ; sa banay ni Levi
(^12) , (^000) ; sa banay ni Isacar
(^12) , (^000) ;
(^8) sa banay ni Zabulon
(^12) , (^000) ; sa banay ni Jose
(^12) , (^000) ; ug sa banay ni Benjamin
(^12) , (^000). Ang Dakong Panon sa Katawhan
```

In [692]:
specific_fix = r'(\n)\(\^(\d{2,3})\) , \(\^(\d{2,3})\)'

print(re.findall(specific_fix, texts['ceb']))
texts['ceb'] = re.sub(specific_fix, r' \2 , \3', texts['ceb'])

print_number_of_lines()
save()

[('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000'), ('\n', '12', '000')]
number of lines:
ceb : 30538
fil : 30421
hil : 30159


In [693]:
texts['ceb'] = re.sub(r'', '', texts['ceb'])
save()

In [694]:
#remove the first line, hanging chapter title
specific_fix = r'Ang Sugilanon sa Kabuhatan\n'
texts['ceb'] = re.sub(specific_fix, '', texts['ceb'])

In [695]:
#remove verses
texts['ceb'] = re.sub(r'^(\d+\s*-\s*\(\^\d+\) |\d+\s*[a-z]\s*-\s*\(\^\d+\) |\(\^\d+\) |\d+ - \d [a-z]|\d+\s*)', '', texts['ceb'], flags=re.MULTILINE)

texts['ceb'] = re.sub(r'\(\^(\d+)\)', r'\1', texts['ceb'])
texts['ceb'] = re.sub(r'(\()\^\d+\s', r'\1', texts['ceb'])

with open('checkpoints/ceb_final_verses.txt', 'w') as f:
    f.write(texts['ceb'])

#segment by sentence
texts['ceb'] = re.sub(r'\n', ' ', texts['ceb'])
texts['ceb'] = re.sub(r'([.?!]’?”?\)?) ?', r'\1\n', texts['ceb'])

with open('checkpoints/ceb_final_sentences.txt', 'w') as f:
    f.write(texts['ceb'])


In [696]:
print("number of words", texts['ceb'].count(' '))

number of words 700982
